# Price/Nero/Gelman radon random-restart VI

This notebook implements a first-pass full-Bayesian version of the peer-reviewed county geometric-mean model from Price, Nero, and Gelman (1996). It uses the same random-restart comparison pattern as the other `single_MC` notebooks and compares final variational summaries to the article's reported posterior means.

Model:

```text
r_adjusted = r_pCi / 2 + sqrt(r_pCi^2 / 4 + 0.25^2)
y_ij = log(37 * r_adjusted_ij)
y_ij ~ Normal(theta_j, sqrt(kappa_sq))
theta_j ~ Normal(mu, sqrt(sigma_sq))
mu ~ Normal(0, 10)
kappa_sq ~ Exponential(1)
sigma_sq ~ Exponential(1)
```

The article reports means for `eta = (mu, kappa_sq, sigma_sq)`: `mu=4.95`, `kappa_sq=0.570`, and `sigma_sq=0.097`. The proper priors above are weak defaults for this PPL implementation, so the comparison should be read as a first-pass benchmark rather than a bit-for-bit reconstruction of the paper's computation.

## PyMC

This version encodes the Price/Nero/Gelman model in PyMC and runs both PyMC default ADVI and PyMC Adam, each with 1 and 100 MC objective samples.

In [ ]:
import os
import gc
from pathlib import Path
os.environ.setdefault('PYTENSOR_FLAGS', 'base_compiledir=/tmp/pytensor')
import sys

sys.path.append(os.path.abspath('../../..'))

import pymc as pm
import pytensor
pytensor.config.cxx = '/usr/bin/clang++'

from modulars.pymc_rr_test import run_pymc_VI
from modulars.radon import build_pymc_price_radon_model

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

from modulars.plot_rr import plot_some_dims_multid
from modulars.radon import (
    PRICE_1996_MODEL_NOTES,
    PRICE_1996_REFERENCE_MEANS_UNCONSTRAINED,
    PRICE_1996_REPORTED_SUMMARIES,
    load_radon_data,
    make_price_radon_param_names,
    plot_radon_selected_dims,
    plot_radon_selected_dims_zoomed,
    price_theta_summary_from_unconstrained,
)
from modulars.utils import apply_traj_transform_multid, save_to_csv

radon_data = load_radon_data()
county_names = radon_data['county_names']
param_names = make_price_radon_param_names(county_names, unconstrained=True)
dim = len(param_names)
which_dims = [0, 1, 2]
which_labels = [param_names[d] for d in which_dims]

print(f"N observations: {len(radon_data['log_radon_bq_adjusted'])}")
print(f"N counties: {len(county_names)}")
print(f"Latent VI dimension: {dim}")
print(list(zip(which_dims, which_labels)))


In [ ]:
# Toggle this before running the notebook.
# "quick" keeps the 100-MC comparison but shortens the run for smoke checks.
# "full" is the research-run setting used for the planned comparison.
RUN_MODE = "full"  # "quick" or "full"

RUN_CONFIGS = {
    "quick": {"max_iters": 250, "n_restarts": 1, "n_particles": 100, "track_every": 10, "save_raw": False},
    "full": {"max_iters": 80_000, "n_restarts": 5, "n_particles": 100, "track_every": 5, "save_raw": False},
}

run_config = RUN_CONFIGS[RUN_MODE]
max_iters = run_config["max_iters"]
n_restarts = run_config["n_restarts"]
n_particles = run_config["n_particles"]
track_every = run_config["track_every"]
save_raw = run_config.get("save_raw", False)

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

run_config


In [ ]:
print('Price/Nero/Gelman reported summaries:')
print(PRICE_1996_REPORTED_SUMMARIES)
print('\nModel notes:')
for key, note in PRICE_1996_MODEL_NOTES.items():
    print(f"- {key}: {note}")

price_reference = pd.read_csv('price_1996_reference_summary.csv')
best_reference_path = Path('best_reference_values_without_pymc.csv')
if not best_reference_path.exists():
    best_reference_path = Path('best_reference_values.csv')
best_reference = pd.read_csv(best_reference_path)

best_reference_means = {
    row['parameter']: [(float(row['mean']), row['source'])]
    for _, row in best_reference.iterrows()
}
best_reference_stds = {
    row['parameter']: [(float(row['sd']), row['source'])]
    for _, row in best_reference.iterrows()
}
reference_means = {key: list(value) for key, value in PRICE_1996_REFERENCE_MEANS_UNCONSTRAINED.items()}
for key, value in best_reference_means.items():
    reference_means.setdefault(key, []).extend(value)
reference_stds = best_reference_stds

(price_reference, best_reference.head())


In [ ]:
def run_model(n_mc_samples=1, optimizer='default', seed=1):
    model = build_pymc_price_radon_model(radon_data)
    try:
        return run_pymc_VI(
            model,
            n_mc_samples=n_mc_samples,
            n_iters=max_iters,
            optimizer=optimizer,
            seed=seed,
            SINGLE_DIM=False,
            track_every=track_every,
            track_cov=False,
        )
    finally:
        del model
        gc.collect()


In [ ]:
# Run all PyMC default-optimizer restarts first so these outputs are saved before Adam starts.
results = []
for seed in tqdm(range(n_restarts)):
    single_mean, single_std = run_model(
        n_mc_samples=1,
        optimizer='default',
        seed=seed,
    )
    multi_mean, multi_std = run_model(
        n_mc_samples=n_particles,
        optimizer='default',
        seed=seed + 1000,
    )
    results.append([single_mean, single_std, multi_mean, multi_std])
    gc.collect()


In [ ]:
if save_raw:
    save_to_csv(results_dir / "pymc_default_raw_restarts.csv", results)


In [ ]:
single_means, single_stds, multi_means, multi_stds = apply_traj_transform_multid(results)
print(single_means.shape, single_stds.shape, multi_means.shape, multi_stds.shape)

save_to_csv(
    results_dir / "pymc_default_processed_restarts.csv",
    [(single_means, single_stds, multi_means, multi_stds)],
)
save_to_csv(
    results_dir / "pymc_default_final_restarts.csv",
    [(multi_means[:, -1, :], multi_stds[:, -1, :])],
)

del results
for _name in ["single_mean", "single_std", "multi_mean", "multi_std"]:
    globals().pop(_name, None)
gc.collect()


In [ ]:
# Now run all Adam-optimizer restarts and save them independently.
results_adam = []
for seed in tqdm(range(n_restarts)):
    adam_single_mean, adam_single_std = run_model(
        n_mc_samples=1,
        optimizer='adam',
        seed=seed,
    )
    adam_multi_mean, adam_multi_std = run_model(
        n_mc_samples=n_particles,
        optimizer='adam',
        seed=seed + 1000,
    )
    results_adam.append([adam_single_mean, adam_single_std, adam_multi_mean, adam_multi_std])
    gc.collect()


In [ ]:
if save_raw:
    save_to_csv(results_dir / "pymc_adam_raw_restarts.csv", results_adam)


In [ ]:
adam_single_means, adam_single_stds, adam_multi_means, adam_multi_stds = apply_traj_transform_multid(results_adam)
print(adam_single_means.shape, adam_single_stds.shape, adam_multi_means.shape, adam_multi_stds.shape)

save_to_csv(
    results_dir / "pymc_adam_processed_restarts.csv",
    [(adam_single_means, adam_single_stds, adam_multi_means, adam_multi_stds)],
)
save_to_csv(
    results_dir / "pymc_adam_final_restarts.csv",
    [(adam_multi_means[:, -1, :], adam_multi_stds[:, -1, :])],
)

del results_adam
for _name in ["adam_single_mean", "adam_single_std", "adam_multi_mean", "adam_multi_std"]:
    globals().pop(_name, None)
gc.collect()


In [ ]:
# load in results
from modulars.utils import load_from_csv
processed_path = results_dir / "pymc_default_processed_restarts.csv"
if processed_path.exists():
    loaded = load_from_csv(processed_path)
    single_means, single_stds, multi_means, multi_stds = loaded[0]


In [ ]:
import importlib
import modulars.radon
importlib.reload(modulars.radon)

In [ ]:
modulars.radon.plot_radon_selected_dims(
    single_means, single_stds, multi_means, multi_stds,
    which_dims, which_labels,
    title_prefix='PyMC default Price ',
    reference_means=reference_means,
    reference_stds=reference_stds,
    iteration_stride=track_every,
)


In [ ]:
modulars.radon.plot_radon_selected_dims_zoomed(
    single_means, single_stds, multi_means, multi_stds,
    which_dims, which_labels,
    title_prefix='PyMC default Price ',
    reference_means=reference_means,
    reference_stds=reference_stds,
    iteration_stride=track_every,
)


In [ ]:
plot_param_names = pymc_param_names if 'pymc_param_names' in globals() else param_names
best_by_parameter = best_reference.set_index('parameter')
best_mean = best_by_parameter.loc[plot_param_names, 'mean'].to_numpy()
best_std = best_by_parameter.loc[plot_param_names, 'sd'].to_numpy()

In [ ]:
plot_some_dims_multid(
    single_means, single_stds, multi_means, multi_stds,
    best_mean, best_std,
    param_name='radon_latent',
    dim=len(plot_param_names),
    which_dims=which_dims,
    k=min(2, n_restarts),
    label_prefix='PyMC default ',
    iteration_stride=track_every,
)

In [ ]:

adam_processed_path = results_dir / "pymc_adam_processed_restarts.csv"
if adam_processed_path.exists():
    loaded_adam = load_from_csv(adam_processed_path)
    adam_single_means, adam_single_stds, adam_multi_means, adam_multi_stds = loaded_adam[0]


In [ ]:
modulars.radon.plot_radon_selected_dims(
    adam_single_means, adam_single_stds, adam_multi_means, adam_multi_stds,
    which_dims, which_labels,
    title_prefix='PyMC Adam Price ',
    reference_means=reference_means,
    reference_stds=reference_stds,
    iteration_stride=track_every,
)


modulars.radon.plot_radon_selected_dims_zoomed(
    adam_single_means, adam_single_stds, adam_multi_means, adam_multi_stds,
    which_dims, which_labels,
    title_prefix='PyMC Adam Price ',
    reference_means=reference_means,
    reference_stds=reference_stds,
    iteration_stride=track_every,
)


In [ ]:



plot_some_dims_multid(
    adam_single_means, adam_single_stds, adam_multi_means, adam_multi_stds,
    best_mean, best_std,
    param_name='radon_latent',
    dim=len(plot_param_names),
    which_dims=which_dims,
    k=min(2, n_restarts),
    label_prefix='PyMC Adam ',
    iteration_stride=track_every,
)


In [ ]:
summary = pd.DataFrame({
    'parameter': param_names,
    'default_single_mc_final_mean': single_means[:, -1, :].mean(axis=0),
    'default_single_mc_final_std': single_stds[:, -1, :].mean(axis=0),
    'default_multi_mc_final_mean': multi_means[:, -1, :].mean(axis=0),
    'default_multi_mc_final_std': multi_stds[:, -1, :].mean(axis=0),
    'adam_single_mc_final_mean': adam_single_means[:, -1, :].mean(axis=0),
    'adam_single_mc_final_std': adam_single_stds[:, -1, :].mean(axis=0),
    'adam_multi_mc_final_mean': adam_multi_means[:, -1, :].mean(axis=0),
    'adam_multi_mc_final_std': adam_multi_stds[:, -1, :].mean(axis=0),
})
summary.iloc[which_dims]


In [ ]:
eta_names = ['mu', 'kappa_sq', 'sigma_sq']

rows = []
for optimizer, means_1, stds_1, means_100, stds_100 in (
    ('default', single_means, single_stds, multi_means, multi_stds),
    ('adam', adam_single_means, adam_single_stds, adam_multi_means, adam_multi_stds),
):
    for setting, means, stds in (
        ('1 MC sample', means_1, stds_1),
        ('100 MC samples', means_100, stds_100),
    ):
        for run in range(means.shape[0]):
            eta_mean, eta_sd = price_theta_summary_from_unconstrained(means[run, -1, :], stds[run, -1, :])
            for parameter, mean_value, sd_value in zip(eta_names, eta_mean, eta_sd):
                rows.append({
                    'optimizer': optimizer,
                    'setting': setting,
                    'run': run,
                    'parameter': parameter,
                    'vi_mean': mean_value,
                    'vi_sd': sd_value,
                })

eta_summary = pd.DataFrame(rows)
eta_final = eta_summary.groupby(['optimizer', 'setting', 'parameter'], as_index=False).agg(
    vi_mean=('vi_mean', 'mean'),
    vi_mean_across_restart_sd=('vi_mean', 'std'),
    vi_sd=('vi_sd', 'mean'),
)
reference_means = price_reference[price_reference['summary'].eq('mean')][['parameter', 'value']]
comparison = eta_final.merge(reference_means, on='parameter', how='left')
comparison['error_vs_price_mean'] = comparison['vi_mean'] - comparison['value']
comparison


In [ ]:
plot_df = comparison.copy()
plot_df['setting'] = pd.Categorical(plot_df['setting'], ['1 MC sample', '100 MC samples'], ordered=True)
plot_df['plot_label'] = plot_df['optimizer'] + '\n' + plot_df['setting'].astype(str)
plot_df = plot_df.sort_values(['parameter', 'optimizer', 'setting'])

fig, axs = plt.subplots(1, len(eta_names), figsize=(5 * len(eta_names), 4), squeeze=False)
for ax, parameter in zip(axs[0], eta_names):
    subset = plot_df[plot_df['parameter'].eq(parameter)].copy()
    x = np.arange(len(subset))
    yerr = subset['vi_mean_across_restart_sd'].fillna(0.0).to_numpy()
    ax.errorbar(
        x,
        subset['vi_mean'],
        yerr=yerr,
        fmt='o',
        capsize=4,
        color='tab:blue',
        label='VI restart mean',
    )
    ax.axhline(float(subset['value'].iloc[0]), color='red', linestyle='--', label='Price et al. 1996')
    ax.set_title(parameter)
    ax.set_xticks(x)
    ax.set_xticklabels(subset['plot_label'], rotation=30, ha='right')
    ax.set_ylabel('constrained-scale value')
    ax.grid(axis='y', alpha=0.35)
    ax.legend()

fig.suptitle('Final PyMC VI summaries vs Price/Nero/Gelman reported posterior means')
plt.tight_layout()
plt.show()


In [ ]:
# Free large trajectory arrays after all summaries and plots are saved.
for _name in ['single_means', 'single_stds', 'multi_means', 'multi_stds', 'best_mean', 'best_std', 'best_by_parameter', 'adam_single_means', 'adam_single_stds', 'adam_multi_means', 'adam_multi_stds']:
    globals().pop(_name, None)
gc.collect()
